In [19]:
import os
import boto3
import sagemaker
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

# Define parameters
bucket = "s3://sagemaker-us-east-1-993768311527"
prefix = "cardio_logistic_pipeline"
role = sagemaker.get_execution_role()
sagemaker_session = sagemaker.session.Session()

# Parameter inputs
data_uri = ParameterString(name="InputData", default_value=f"{bucket}/{prefix}/data/cardio_train.csv")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="Approved")


### Step 1: Run Preprocessing Job

In [14]:
# Step 1: Preprocessing
processor = SKLearnProcessor(
    framework_version="0.23-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="cardio-preprocess",
    sagemaker_session=sagemaker_session
)

step_process = ProcessingStep(
    name="CardioDataProcessing",
    processor=processor,
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=data_uri,
            destination="/opt/ml/processing/input",
            input_name="input-1"
        )
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="train_data",
            source="/opt/ml/processing/train"
        ),
        sagemaker.processing.ProcessingOutput(
            output_name="test_data",
            source="/opt/ml/processing/test"
        )
    ],
    code="preprocessing.py"
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


### Step 2: Train Logistic Regression Model

In [15]:
# Step 2: Training
sklearn = SKLearn(
    entry_point="train_logistic.py",
    role=role,
    instance_type="ml.m5.large",
    framework_version="0.23-1",
    base_job_name="cardio-logistic-train",
    sagemaker_session=sagemaker_session
)

step_train = TrainingStep(
    name="TrainLogisticModel",
    estimator=sklearn,
    inputs={
        "train": sagemaker.inputs.TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train_data"].S3Output.S3Uri,
            content_type="text/csv"
        )
    },
    cache_config=CacheConfig(enable_caching=True, expire_after="1d")
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/steps.py:485: UserWarning: Profiling is enabled on the provided estimator. The default profiler rule includes a timestamp which will change each time the pipeline is upserted, causing cache misses. If profiling is not needed, set disable_profiler to True on the estimator.
  warnings.warn(msg)


### ✅ Step 3: Evaluate the Trained Model

In [16]:
# Step 3: Evaluation
eval_processor = SKLearnProcessor(
    framework_version="0.23-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="cardio-eval",
    sagemaker_session=sagemaker_session
)

eval_property_file = PropertyFile(name="EvaluationReport", output_name="evaluation", path="/opt/ml/processing/evaluation/evaluation.json")

step_eval = ProcessingStep(
    name="EvaluateLogisticModel",
    processor=eval_processor,
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        sagemaker.processing.ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test_data"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        )
    ],
    code="evaluate.py",
    property_files=[eval_property_file]
)


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


### ✅ Step 4: Register Model Conditionally Based on AUC

In [21]:
# Step 4: Conditional Registration
from sagemaker.workflow.step_collections import RegisterModel

register_step = RegisterModel(
    name="RegisterLogisticModel",
    estimator=sklearn,  # reuses your existing SKLearn estimator from step 2
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="CardioLogisticModelGroup",
    approval_status=model_approval_status,
    sagemaker_session=sagemaker_session
)

cond_gte = ConditionGreaterThanOrEqualTo(
    left=JsonGet(step_name=step_eval.name, property_file=eval_property_file, json_path="metrics.auc"),
    right=0.75
)

step_cond = ConditionStep(
    name="CheckAUCThreshold",
    conditions=[cond_gte],
    if_steps=[register_step],
    else_steps=[]
)

# Define Pipeline
pipeline = Pipeline(
    name="CardioLogisticPipeline",
    parameters=[data_uri, model_approval_status],
    steps=[step_process, step_train, step_eval, step_cond],
    sagemaker_session=sagemaker_session
)

pipeline.upsert(role_arn=role)
print("✅ CI/CD pipeline for logistic regression created and registered.")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:37                                                                                   │
│                                                                                                  │
│   34 │   sagemaker_session=sagemaker_session                                                     │
│   35 )                                                                                           │
│   36                                                                                             │
│ ❱ 37 pipeline.upsert(role_arn=role)                                                              │
│   38 print("✅ CI/CD pipeline for logistic regression created and registered.")                  │
│   39                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:292 in upsert             │
│                                                                                                  │
│    289 │   │   │   # after fetching the config.                                                  │
│    290 │   │   │   raise ValueError("An AWS IAM role is required to create or update a Pipeline  │
│    291 │   │   try:                                                                              │
│ ❱  292 │   │   │   response = self.create(role_arn, description, tags, parallelism_config)       │
│    293 │   │   except ClientError as ce:                                                         │
│    294 │   │   │   error_code = ce.response["Error"]["Code"]                                     │
│    295 │   │   │   error_message = ce.response["Error"]["Message"]                               │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:164 in create             │
│                                                                                                  │
│    161 │   │   tags = format_tags(tags)                                                          │
│    162 │   │   tags = _append_project_tags(tags)                                                 │
│    163 │   │   tags = self.sagemaker_session._append_sagemaker_config_tags(tags, PIPELINE_TAGS_  │
│ ❱  164 │   │   kwargs = self._create_args(role_arn, description, parallelism_config)             │
│    165 │   │   update_args(                                                                      │
│    166 │   │   │   kwargs,                                                                       │
│    167 │   │   │   Tags=tags,                                                                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:186 in _create_args       │
│                                                                                                  │
│    183 │   │   Returns:                                                                          │
│    184 │   │   │   A keyword argument dict for calling create_pipeline.                          │
│    185 │   │   """                                                                               │
│ ❱  186 │   │   pipeline_definition = self.definition()                                           │
│    187 │   │   kwargs = dict(                                                                    │
│    188 │   │   │   PipelineName=self.name,                                                       │
│    189 │   │   │   RoleArn=role_arn,                                                             │
│                                                             

In [8]:
import boto3

s3 = boto3.client("s3")

# Update these paths
bucket = "sagemaker-us-east-1-993768311527"
s3_key = "cardio_logistic_pipeline/data/cardio_train.csv"
local_file = "./cardio_train.csv"

# Upload to S3
s3.upload_file(local_file, bucket, s3_key)

print("✅ cardio_train.csv uploaded to:", f"s3://{bucket}/{s3_key}")

✅ cardio_train.csv uploaded to: s3://sagemaker-us-east-1-993768311527/cardio_logistic_pipeline/data/cardio_train.csv


In [7]:
pipeline.start()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 pipeline.start()                                                                             │
│   2                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:376 in start              │
│                                                                                                  │
│    373 │   │   update_args(kwargs, PipelineParameters=format_start_parameters(parameters))       │
│    374 │   │                                                                                     │
│    375 │   │   # retry on AccessDeniedException to cover case of tag propagation delay           │
│ ❱  376 │   │   response = retry_with_backoff(                                                    │
│    377 │   │   │   lambda: self.sagemaker_session.sagemaker_client.start_pipeline_execution(**k  │
│    378 │   │   │   botocore_client_error_code="AccessDeniedException",                           │
│    379 │   │   )                                                                                 │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/utils.py:753 in retry_with_backoff             │
│                                                                                                  │
│    750 │   │   │   │   if i == num_attempts - 1:                                                 │
│    751 │   │   │   │   │   raise ex                                                              │
│    752 │   │   │   else:                                                                         │
│ ❱  753 │   │   │   │   raise ex                                                                  │
│    754 │   │   │   logger.error("Retrying in attempt %s, due to %s", (i + 1), str(ex))           │
│    755 │   │   │   time.sleep(2**i)                                                              │
│    756                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/utils.py:742 in retry_with_backoff             │
│                                                                                                  │
│    739 │   │   )                                                                                 │
│    740 │   for i in range(num_attempts):                                                         │
│    741 │   │   try:                                                                              │
│ ❱  742 │   │   │   return callable_func()                                                        │
│    743 │   │   except Exception as ex:  # pylint: disable=broad-except                           │
│    744 │   │   │   if not botocore_client_error_code or (                                        │
│    745 │   │   │   │   botocore_client_error_code                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:377 in <lambda>           │
│                                                                                                  │
│    374 │   │                                                                                     │
│    375 │   │   # retry on AccessDeniedException to cover ca